# Adaptive Dynamic RFM — Customer Behavior Analysis & Prediction

**Đề tài:** Phân tích và dự báo hành vi khách hàng sử dụng mô hình Dynamic RFM.

Notebook triển khai:
- Cleaning dữ liệu giao dịch Online Retail II.
- Monthly snapshots và nhãn **mua hàng trong tháng kế tiếp**.
- Temporal Train / Validation / Test split.
- Baseline Lifetime/Static RFM và Fixed Rolling RFM.
- Proposed method: Adaptive Rolling Window + Temporal Features.
- So sánh Logistic Regression, Random Forest, XGBoost (nếu cài đặt).
- Chỉ dùng Validation để lựa chọn window/model; Test chỉ dùng đánh giá cuối cùng.

> Lưu ý: Đây là notebook nghiên cứu thực nghiệm. Các kết quả sẽ được tính khi chạy trên dataset thực tế; không có metric giả lập được ghi sẵn.


## 1. Import libraries & configuration

In [ ]:
import warnings, os, glob
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
DATA_PATH = None  # Set path, e.g. r"/content/online_retail_ii.xlsx"
WINDOWS = [30, 60, 90, 120, 180]
SNAPSHOT_START = "2010-01"
SNAPSHOT_END = "2011-10"  # October snapshot predicts November (complete month)
TRAIN_END = "2010-12"
VALID_START, VALID_END = "2011-01", "2011-06"
TEST_START, TEST_END = "2011-07", "2011-10"

pd.set_option("display.max_columns", 100)


## 2. Load Online Retail II
Upload the Excel file or set `DATA_PATH`. The loader reads both sheets when available.

In [ ]:
def find_dataset():
    if DATA_PATH and os.path.exists(DATA_PATH):
        return DATA_PATH
    candidates = (
        glob.glob("**/*online*retail*ii*.xlsx", recursive=True) +
        glob.glob("**/*retail*ii*.xlsx", recursive=True) +
        glob.glob("**/*.xlsx", recursive=True)
    )
    return candidates[0] if candidates else None

path = find_dataset()
if path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        path = next(iter(uploaded.keys()))
    except Exception:
        raise FileNotFoundError("Set DATA_PATH to the Online Retail II Excel file.")

print("Dataset:", path)

xls = pd.ExcelFile(path)
frames = []
for sheet in xls.sheet_names:
    tmp = pd.read_excel(path, sheet_name=sheet)
    frames.append(tmp)
df_raw = pd.concat(frames, ignore_index=True)
print("Raw shape:", df_raw.shape)
display(df_raw.head())


## 3. Data Understanding

In [ ]:
display(df_raw.info())
display(df_raw.isna().sum().sort_values(ascending=False))
display(df_raw.describe(include="all").T)


## 4. Basic Data Cleaning
Cleaning rules are based on record validity, not statistics learned from the entire dataset.

In [ ]:
df = df_raw.copy()

# Normalize column names
df.columns = [str(c).strip() for c in df.columns]
rename_map = {
    "Invoice": "Invoice",
    "StockCode": "StockCode",
    "Description": "Description",
    "Quantity": "Quantity",
    "InvoiceDate": "InvoiceDate",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID",
    "CustomerID": "CustomerID",
    "Country": "Country"
}
df = df.rename(columns=rename_map)

required = ["Invoice", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Keep valid customer transactions
df = df.dropna(subset=["InvoiceDate", "CustomerID", "Invoice", "Quantity", "UnitPrice"])
df["CustomerID"] = df["CustomerID"].astype("int64")
df["Invoice"] = df["Invoice"].astype(str).str.strip()

# Cancelled invoices usually have invoice IDs beginning with C
df = df[~df["Invoice"].str.upper().str.startswith("C", na=False)]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

# Remove exact duplicate rows
df = df.drop_duplicates()

df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]
df = df.sort_values("InvoiceDate").reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Customers:", df["CustomerID"].nunique())
print("Invoices:", df["Invoice"].nunique())
display(df.head())


## 5. Explore transaction volume

In [ ]:
monthly_tx = df.set_index("InvoiceDate").resample("MS").agg(
    Transactions=("Invoice", "nunique"),
    Customers=("CustomerID", "nunique"),
    Revenue=("TotalAmount", "sum")
)
display(monthly_tx)
monthly_tx[["Transactions", "Customers", "Revenue"]].plot(
    subplots=True, figsize=(12, 8), title="Monthly Transaction Overview"
)
plt.tight_layout()
plt.show()


## 6. Temporal split & target definition

Each snapshot at month-end uses transactions up to that cutoff. The target is whether a previously known customer purchases during the **following calendar month**.

- Train snapshots: 2010-01 to 2010-12
- Validation snapshots: 2011-01 to 2011-06
- Test snapshots: 2011-07 to 2011-10

The October 2011 snapshot predicts November 2011, avoiding the incomplete December period in this dataset.


In [ ]:
def month_periods(start=SNAPSHOT_START, end=SNAPSHOT_END):
    return pd.period_range(start=start, end=end, freq="M")

snapshot_periods = month_periods()
print(snapshot_periods)


## 7. Snapshot generator: RFM + next-month purchase label

In [ ]:
def make_snapshot(data, period, window_days=None):
    period = pd.Period(period, freq="M")
    cutoff = period.end_time.normalize() + pd.Timedelta(days=1)  # exclusive cutoff
    next_start = (period + 1).start_time
    next_end = (period + 1).end_time + pd.Timedelta(days=1)      # exclusive

    hist = data[data["InvoiceDate"] < cutoff]
    future = data[(data["InvoiceDate"] >= next_start) &
                  (data["InvoiceDate"] < next_end)]

    # Customers known before snapshot
    customers = hist["CustomerID"].unique()
    if len(customers) == 0:
        return pd.DataFrame()

    last_purchase = hist.groupby("CustomerID")["InvoiceDate"].max()
    recency = (cutoff - last_purchase).dt.days.rename("R")

    if window_days is None:
        selected = hist
        suffix = "lifetime"
    else:
        start_date = cutoff - pd.Timedelta(days=window_days)
        selected = hist[hist["InvoiceDate"] >= start_date]
        suffix = f"{window_days}d"

    agg = selected.groupby("CustomerID").agg(
        F=("Invoice", "nunique"),
        M=("TotalAmount", "sum"),
        Q=("Quantity", "sum")
    )

    out = pd.DataFrame(index=pd.Index(customers, name="CustomerID"))
    out = out.join(recency)
    out = out.join(agg)
    out[["F", "M", "Q"]] = out[["F", "M", "Q"]].fillna(0)
    out["snapshot"] = period.to_timestamp("M")
    out["snapshot_period"] = str(period)
    out["window"] = suffix

    future_customers = set(future["CustomerID"].unique())
    out["target_purchase_next_month"] = out.index.isin(future_customers).astype(int)
    out = out.reset_index()
    return out

# Quick sanity check
sample = make_snapshot(df, "2010-06", 90)
display(sample.head())
print("Snapshot rows:", len(sample), "| Positive target rate:", sample["target_purchase_next_month"].mean())


## 8. Build datasets for all candidate windows

In [ ]:
snapshot_cache = {}
for window in WINDOWS:
    parts = [make_snapshot(df, p, window) for p in snapshot_periods]
    data_window = pd.concat(parts, ignore_index=True)
    snapshot_cache[window] = data_window
    print(window, "days:", data_window.shape, "positive rate:", round(data_window["target_purchase_next_month"].mean(), 4))

# Lifetime/static RFM baseline
static_parts = [make_snapshot(df, p, None) for p in snapshot_periods]
static_data = pd.concat(static_parts, ignore_index=True)


## 9. Temporal feature engineering
Lag features are calculated only from earlier snapshots for the same customer.

In [ ]:
BASE_FEATURES = ["R", "F", "M", "Q"]

def add_temporal_features(data):
    x = data.sort_values(["CustomerID", "snapshot"]).copy()
    for col in BASE_FEATURES:
        x[f"{col}_lag1"] = x.groupby("CustomerID")[col].shift(1)
        x[f"{col}_lag2"] = x.groupby("CustomerID")[col].shift(2)
        x[f"{col}_delta1"] = x[col] - x[f"{col}_lag1"]
    x["F_trend_3"] = x.groupby("CustomerID")["F"].transform(
        lambda s: s - s.shift(2)
    )
    x["M_trend_3"] = x.groupby("CustomerID")["M"].transform(
        lambda s: s - s.shift(2)
    )
    # Missing historical snapshots mean "no previous snapshot"; imputation happens in model pipeline.
    return x

feature_cache = {w: add_temporal_features(data) for w, data in snapshot_cache.items()}
static_features = add_temporal_features(static_data)
display(feature_cache[90].head())


## 10. Define temporal Train / Validation / Test

In [ ]:
def split_temporal(data):
    p = data["snapshot_period"]
    train = data[p <= TRAIN_END].copy()
    valid = data[(p >= VALID_START) & (p <= VALID_END)].copy()
    test = data[(p >= TEST_START) & (p <= TEST_END)].copy()
    return train, valid, test

for w, data in feature_cache.items():
    tr, va, te = split_temporal(data)
    print(f"{w}d -> train={tr.shape}, validation={va.shape}, test={te.shape}")

train_static, valid_static, test_static = split_temporal(static_features)


## 11. Feature columns & evaluation functions

In [ ]:
FEATURES = (
    BASE_FEATURES +
    [f"{c}_lag1" for c in BASE_FEATURES] +
    [f"{c}_lag2" for c in BASE_FEATURES] +
    [f"{c}_delta1" for c in BASE_FEATURES] +
    ["F_trend_3", "M_trend_3"]
)
TARGET = "target_purchase_next_month"

def get_xy(data):
    X = data[FEATURES].replace([np.inf, -np.inf], np.nan)
    y = data[TARGET].astype(int)
    return X, y

def make_lr():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
    ])

def make_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=250, max_depth=None, min_samples_leaf=2,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])

def metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    result = {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "PR_AUC": average_precision_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan
    }
    return result


## 12. Baseline comparison: Static RFM & Fixed Rolling Windows
Use Random Forest consistently for this first comparison. Window selection is based on Validation F1; Test remains untouched.

In [ ]:
baseline_results = []
models_by_window = {}

# Static / lifetime baseline
for name, data in [("Static/Lifetime RFM", static_features)] + [
    (f"Fixed Rolling {w}d", feature_cache[w]) for w in WINDOWS
]:
    tr, va, te = split_temporal(data)
    Xtr, ytr = get_xy(tr)
    Xva, yva = get_xy(va)
    model = make_rf()
    model.fit(Xtr, ytr)
    pva = model.predict_proba(Xva)[:, 1]
    score = metrics(yva, pva)
    baseline_results.append({"Method": name, **score})
    models_by_window[name] = model

baseline_validation = pd.DataFrame(baseline_results).sort_values("F1", ascending=False)
display(baseline_validation)


## 13. Adaptive Rolling Window Selection
Choose the window with the highest Validation F1. This selection does not use Test labels.

In [ ]:
adaptive_scores = baseline_validation[
    baseline_validation["Method"].str.startswith("Fixed Rolling")
].copy()

best_method = adaptive_scores.iloc[0]["Method"]
best_window = int(best_method.split()[-1].replace("d", ""))
print("Selected adaptive window:", best_window, "days")
print("Validation F1:", adaptive_scores.iloc[0]["F1"])


## 14. Proposed method: Adaptive RFM + Temporal Features
Train the selected window's model using Train and evaluate on Validation.

In [ ]:
best_data = feature_cache[best_window]
train_best, valid_best, test_best = split_temporal(best_data)

X_train, y_train = get_xy(train_best)
X_valid, y_valid = get_xy(valid_best)
X_test, y_test = get_xy(test_best)

adaptive_rf = make_rf()
adaptive_rf.fit(X_train, y_train)
valid_prob = adaptive_rf.predict_proba(X_valid)[:, 1]
print("Adaptive Validation:", metrics(y_valid, valid_prob))


## 15. Model comparison on Validation
Compare Logistic Regression, Random Forest and XGBoost when installed.

In [ ]:
candidate_models = {
    "Logistic Regression": make_lr(),
    "Random Forest": make_rf()
}
try:
    from xgboost import XGBClassifier
    candidate_models["XGBoost"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
except Exception as e:
    print("XGBoost unavailable; skipped:", e)

model_validation_rows = []
fitted_models = {}
for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    p = model.predict_proba(X_valid)[:, 1]
    model_validation_rows.append({"Model": name, **metrics(y_valid, p)})
    fitted_models[name] = model

model_validation = pd.DataFrame(model_validation_rows).sort_values("F1", ascending=False)
display(model_validation)
best_model_name = model_validation.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
print("Best validation model:", best_model_name)


## 16. Final Test evaluation
The Test set is evaluated only after window/model selection is completed.

In [ ]:
test_prob = best_model.predict_proba(X_test)[:, 1]
test_result = metrics(y_test, test_prob)
print("Final Test Metrics")
display(pd.DataFrame([test_result], index=[best_model_name]))

test_pred = (test_prob >= 0.5).astype(int)
print(classification_report(y_test, test_pred, zero_division=0))
print("Confusion matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, test_pred),
    index=["Actual No Purchase", "Actual Purchase"],
    columns=["Predicted No Purchase", "Predicted Purchase"]
))


## 17. Baseline vs Proposed on Test
For a fair comparison, evaluate fixed 90-day RFM and the validation-selected adaptive window on the same Test period.

In [ ]:
test_comparison_rows = []

# Static baseline
for name, data in [("Static/Lifetime RFM", static_features),
                    ("Fixed Rolling 90d", feature_cache[90]),
                    (f"Adaptive Rolling {best_window}d", feature_cache[best_window])]:
    tr, va, te = split_temporal(data)
    Xt, yt = get_xy(tr)
    Xte, yte = get_xy(te)
    model = make_rf()
    model.fit(Xt, yt)
    p = model.predict_proba(Xte)[:, 1]
    test_comparison_rows.append({"Method": name, **metrics(yte, p)})

# Proposed method uses selected window + temporal features (same feature construction)
tr, va, te = split_temporal(feature_cache[best_window])
Xt, yt = get_xy(tr)
Xte, yte = get_xy(te)
proposed_rf = make_rf()
proposed_rf.fit(Xt, yt)
p = proposed_rf.predict_proba(Xte)[:, 1]
test_comparison_rows.append({"Method": "Proposed Adaptive RFM + Temporal Features", **metrics(yte, p)})

test_comparison = pd.DataFrame(test_comparison_rows)
display(test_comparison.sort_values("F1", ascending=False))


## 18. Visualize Test comparison

In [ ]:
plot_df = test_comparison.set_index("Method")[["F1", "ROC_AUC", "PR_AUC"]]
plot_df.plot(kind="bar", figsize=(12, 5))
plt.title("Baseline vs Proposed — Test Metrics")
plt.ylabel("Score")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## 19. Customer behavior analysis example

In [ ]:
# Inspect the latest available test snapshot and customer predictions.
latest_test_snapshot = test_best["snapshot"].max()
latest = test_best[test_best["snapshot"] == latest_test_snapshot].copy()
latest_X = latest[FEATURES].replace([np.inf, -np.inf], np.nan)
latest["purchase_probability"] = best_model.predict_proba(latest_X)[:, 1]
latest["predicted_purchase"] = (latest["purchase_probability"] >= 0.5).astype(int)

display(latest.sort_values("purchase_probability", ascending=False).head(20))


## 20. Research interpretation checklist

Khi báo cáo kết quả, cần trả lời:
1. Adaptive window có cải thiện F1/PR-AUC trên Validation và Test so với Fixed Rolling RFM không?
2. Temporal features có giúp dự báo tốt hơn RFM tại snapshot hiện tại không?
3. Model nào phù hợp nhất với dữ liệu?
4. Tỷ lệ positive/negative có mất cân bằng không? Nếu có, F1/PR-AUC quan trọng hơn Accuracy.
5. Có kiểm tra leakage không? Snapshot chỉ sử dụng dữ liệu trước cutoff; target lấy từ tháng kế tiếp.
6. Không tuyên bố cải tiến nếu Test metrics không hỗ trợ.

**Giới hạn:** Notebook hiện dự báo hành vi *có mua trong tháng tiếp theo hay không*. Đây là một mục tiêu cụ thể, kiểm chứng được và phù hợp với tên đề tài. Các mục tiêu như CLV, churn dài hạn hoặc segment-transition prediction có thể là nghiên cứu mở rộng, không nên gộp tất cả vào phiên bản đầu.
